## Nome: Felipe Tagawa Reis
## Matricula: 2037
-----


# **ATV — Testes Categorizados**

Esta atividade pratica os **testes para dados categorizados**: Qui‑quadrado clássico, Qui‑quadrado com correção de Yates, Teste Exato de Fisher e Teste de McNemar.

**Objetivos**
- Montar tabelas de contingência e formular hipóteses (H0/H1).
- Calcular valores **esperados** e a estatística **χ²** (ou **p‑valor**), decidir e concluir.
- Usar `scipy.stats`/`statsmodels` para conferir os cálculos.
- Interpretar resultados nos níveis usuais de significância (1%, 5%, 10%).




## Instruções gerais

- Execute as células na ordem em que aparecem.
- Sempre **declare as hipóteses** antes de calcular (H0 = igualdade; H1 = diferença).
- Para tabelas 2×2, você pode usar:
  - `stats.chi2_contingency(obs, correction=False)` → χ² clássico;
  - `stats.chi2_contingency(obs, correction=True)` → com **Yates**;
  - `stats.fisher_exact(obs, alternative="two-sided")` → **Fisher** (n pequeno ou valores esperados < 5);
  - `mcnemar(tabela_pareada, exact=True/False, correction=True/False)` → **McNemar** (amostras pareadas).

In [19]:
# Imports úteis para toda a atividade
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

pd.set_option("display.precision", 4)



## Funções auxiliares


In [ ]:
def esperados_from_obs(obs):
    obs = np.asarray(obs, dtype=float)
    row_sum = obs.sum(axis=1, keepdims=True)
    col_sum = obs.sum(axis=0, keepdims=True)
    total = obs.sum()
    return row_sum @ col_sum / total

def decide_from_p(p, alphas=(0.10, 0.05, 0.01)):
    return {a: ("Rejeita H0" if p < a else "Não rejeita H0") for a in alphas}

def decide_from_chi2(chi2, crits):
    return {a: ("Rejeitar H0" if chi2 > crits[a] else "Falha em rejeitar H0") for a in crits}




## Exercício 1 — Qui‑quadrado **clássico** (amostras independentes, **n > 40**)

**Contexto**: Ensaio clínico com pacientes com dor abdominal. Grupo **Tratamento** recebeu brometo de pinavério; grupo **Controle** recebeu placebo.  
Tabela de **valores observados** (2×2), **n = 154**:

|                 | **Dor Sim** | **Dor Não** | **Total** |
|-----------------|:-----------:|:-----------:|:---------:|
| **Tratamento**  |      6      |     57      |    63     |
| **Controle**    |     30      |     61      |    91     |
| **Total**       |     36      |    118      |   154     |

1. Formule H0 e H1.
2. Calcule a **tabela de esperados** e a estatística **χ² calculado** (sem correção).
3. Obtenha **gl**, **χ² tabelado** para α∈{10%,5%,1%} (use SciPy) e o **p‑valor**.
4. Decida e conclua sobre a **eficácia do tratamento**.


In [21]:

# Tabela observada (Exemplo 1)
obs1 = np.array([[6, 57],
                 [30, 61]])
obs1_df = pd.DataFrame(obs1, index=["Tratamento","Controle"], columns=["Dor_Sim","Dor_Nao"])


In [22]:

# χ² clássico (sem correção de Yates) e esperados - Questão 2
expected = esperados_from_obs(obs1)
chi2 = ((obs1 - expected)**2 / expected).sum()

print(f"O valor de X^2: {chi2.round(3)}, os valores esperados são: {expected.round(3)}")



O valor de X^2: 11.422, os valores esperados são: [[14.727 48.273]
 [21.273 69.727]]


In [ ]:

# χ² tabelado (crítico)

alphas = [0.10, 0.05, 0.01]

# Questão 3

numLinhas, numColunas = obs1.shape

gl = (numLinhas - 1) * (numColunas - 1)
print(f"Graus de liberdade: {gl}")

chi2_crit = {a: stats.chi2.ppf(1 - a, gl) for a in alphas}

print("Valores Críticos (χ² tabelado) para gl = 1")

for alpha, critical_value in chi2_crit.items():
    print(f"α={alpha*100:.0f}%: {critical_value:.3f}")


p_valor = stats.chi2.sf(chi2, gl)

decisoes = decide_from_p(p_valor)

print() # Quebra linha

print("Decisão Estatística (Baseado no p-valor):")
for alpha, decisao in decisoes.items():
    print(f"Para α={alpha*100:.0f}%: {decisao}")

decisoes_chi2 = decide_from_chi2(chi2, chi2_crit)
print("\nDecisões (χ² x χ²_crit):")
for a, d in decisoes_chi2.items():
    print(f"  α={a:.2f}: {d}")


Graus de liberdade: 1
Valores Críticos (χ² tabelado) para gl = 1
α=10%: 2.706
α=5%: 3.841
α=1%: 6.635

Decisão Estatística (Baseado no p-valor):
Para α=10%: Rejeita H0
Para α=5%: Rejeita H0
Para α=1%: Rejeita H0

Decisões (χ² x χ²_crit):
  α=0.10: Rejeitar H0
  α=0.05: Rejeitar H0
  α=0.01: Rejeitar H0


**Conclusão:**

1.
H0 -> Não há associação entre o tratamento e a presença de dor (tratamento e dor são independentes);
H1 -> Há associação entre o tratamento e a presença de dor (tratamento e dor não são independentes);

4.
A rejeição da hipótese nula ($H_0$) implica que existe uma associação estatisticamente significativa entre o tipo de tratamento e o desfecho da dor (Dor Sim/Dor Não).
O tratamento com brometo de pinavério é considerado Eficaz. Há forte evidência estatística de que o tratamento resultou em uma proporção significativamente maior de pacientes com resolução da dor em comparação com o grupo placebo.



## Exercício 2 — Qui‑quadrado **com correção de Yates** (amostras independentes, **20 < n < 40**)

Considere a tabela 2×2 abaixo com **n = 32** (dados fictícios para prática).

| Grupo | Sucesso | Fracasso | Total |
|-------|:-------:|:--------:|:-----:|
| A     |   14    |    2     |  16   |
| B     |    6    |   10     |  16   |
| **Total** | 20 | 12 | 32 |

1. Formule H0 e H1.
2. Calcule os **esperados** e **χ²** com **correção de Yates**.
3. Informe **gl**, **p‑valor** e **decisão** para α∈{10%,5%}.


In [38]:

obs2 = np.array([[14, 2],
                 [ 6,10]])

exp2 = esperados_from_obs(obs2)
# χ² com correção de Yates
chi2_y = (((np.abs(obs2 - exp2) - 0.5)**2) / exp2).sum()

# Questão 2

print(f"O valor de X^2: {chi2_y.round(3)}, os valores esperados são: {exp2.astype(int)}") # Sumir com o '.'



O valor de X^2: 6.533, os valores esperados são: [[10  6]
 [10  6]]


In [ ]:

# Valores críticos desejados
alphasDesejados = [0.10, 0.05]

# Graus de liberdade
gl = (numLinhas - 1) * (numColunas - 1)
print(f"Graus de liberdade: {gl}")

# χ² crítico
crit2 = {a: stats.chi2.ppf(1 - a, gl) for a in alphasDesejados}

print("Valores Críticos (χ² tabelado) para gl = 1")
for alpha, valores in crit2.items():
    print(f"α={alpha*100:.0f}%: {valores:.3f}")

# p-valor do χ² com Yates
p_valor = stats.chi2.sf(chi2_y, gl)

print("\nDecisão Estatística (Baseado no p-valor):")
decisoes = decide_from_p(p_valor, alphas=alphasDesejados)
for alpha, decisao in decisoes.items():
    print(f"Para α={alpha*100:.0f}%: {decisao}")

# Decisão pelo valor crítico
decisoes_chi2 = decide_from_chi2(chi2_y, crit2)

print("\nDecisões (χ² x χ² crítico):")
for alpha, decisao in decisoes_chi2.items():
    print(f"Para α={alpha*100:.0f}%: {decisao}")



Graus de liberdade: 1
Valores Críticos (χ² tabelado) para gl = 1
α=10%: 2.706
α=5%: 3.841

Decisão Estatística (Baseado no p-valor):
Para α=10%: Rejeita H0
Para α=5%: Rejeita H0

Decisões (χ² x χ² crítico):
Para α=10%: Rejeitar H0
Para α=5%: Rejeitar H0




**Conclusão:**
1.
Hipótese Nula ($H_0$): Não há associação entre o grupo (A ou B) e o desfecho (Sucesso ou Fracasso). As proporções de sucesso são as mesmas nos Grupos A e B;Hipótese Alternativa ($H_1$): Há associação entre o grupo e o desfecho. As proporções de sucesso são diferentes nos Grupos A e B.

Como o $p$-valor ($0.0106$) é menor que $\alpha = 0.05$, há evidência estatística para rejeitar a hipótese nula, evidenciando que Sucesso e Fracasso tem relação.



## Exercício 3 — **Teste Exato de Fisher** (amostras independentes, **n < 20** ou esperados < 5)

**Sobrevida de ratos** após 1,5 ano:

|           | **Vivos** | **Mortos** | **Total** |
|-----------|:---------:|:----------:|:---------:|
| **Normal**      |     5     |     2      |    7     |
| **Experimental**|     1     |     8      |    9     |
| **Total**       |     6     |    10      |    16    |

1. Formule H0 e H1 (duas caudas).
2. Aplique **Fisher** (two‑sided) → **p‑valor**.
3. Decida em α=5% e conclua.


In [26]:

obs3 = np.array([[5,2],
                 [1,8]])
oddsratio, p_fisher = stats.fisher_exact(obs3, alternative="two-sided")

alphaFisher = 0.05

print(oddsratio)
print(p_fisher)

decide_from_p(p_fisher, alphas=[alphaFisher])

20.0
0.03496503496503496


{0.05: 'Rejeita H0'}


**Conclusão:**

Hipótese Nula ($H_0$): As proporções de sobrevivência são iguais nos dois grupos.Hipótese Alternativa ($H_1$): As proporções de sobrevivência são diferentes entre os grupos.

Uma vez que o $p$-valor ($0.03496$) é menor que $\alpha$ ($0.05$), a decisão é rejeitar a Hipótese Nula ($H_0$). Há evidências estatísticas de que as proporções de sobrevivência diferem entre os grupos.


## Exercício 4 — **Teste de McNemar** (amostras pareadas: antes × depois)

**Teste de memória (n = 40)** — número de acertos **completos** (15 palavras) antes e depois do fármaco:

Tabela **pareada** (linhas = Antes; colunas = Depois):

|             | **Correto** | **Incorreto** | **Total** |
|-------------|:-----------:|:-------------:|:---------:|
| **Correto** |      8      |       1       |     9     |
| **Incorreto** |    14     |      17       |    31     |
| **Total**   |     22      |      18       |    40     |

1. Formule H0 e H1 (duas caudas).
2. Calcule McNemar com e sem correção de Yates; reporte **estatística** e **p‑valor**.
3. Decida (α=1% e α=5%) e conclua.


In [27]:

tab = np.array([[8,1],[14,17]])
res_semc = mcnemar(tab, exact=False, correction=False)

chi2_semc = res_semc.statistic # Cast para Float
p_semc = res_semc.pvalue

# Questão 2

print(f"Estatística χ²: {chi2_semc:.3f}")
print(f"p-valor: {p_semc:.6f}")

# Questão 3

decide_from_p(p_semc, alphas=[0.01, 0.05])


Estatística χ²: 11.267
p-valor: 0.000789


{0.01: 'Rejeita H0', 0.05: 'Rejeita H0'}

**Conclusão:**

a Hipótese Nula ($H_0$) estabelece que o fármaco não tem efeito, ou seja, a proporção de indivíduos que mudam para "Correto" é igual à proporção de indivíduos que mudam para "Incorreto" ($P(\text{melhora}) = P(\text{piora})$); já a Hipótese Alternativa ($H_1$) afirma que o fármaco tem um efeito, indicando que a proporção de mudanças em uma direção é significativamente diferente da proporção na direção oposta ($P(\text{melhora}) \neq P(\text{piora})$)

A rejeição de $H_0$ indica que o fármaco tem um efeito estatisticamente significativo na memória dos pacientes.